In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr')

df = load_dataset('크루즈입항정보')

In [11]:
# ============================================================
# 1. 크루즈 승객 데이터 추출 및 전처리
# ============================================================

passenger_data = df[
    (df["입출"] == "입항") &
    (df["선박용도"] == "크루즈선")
].copy()

passenger_data["입항일시"] = pd.to_datetime(
    passenger_data["입항일시"],
    errors="coerce"
)

passenger_data["승객"] = pd.to_numeric(
    passenger_data["승객"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)

passenger_data = passenger_data.dropna(
    subset=["입항일시"]
).copy()

passenger_data["연도"] = passenger_data["입항일시"].dt.year


# ============================================================
# 2. 연도별 승객 수 집계
# ============================================================

annual_passengers = (
    passenger_data.groupby("연도")["승객"]
    .sum(min_count=1)
    .reset_index(name="승객수")
)

# 전체 연도 범위 생성
all_years = range(
    int(annual_passengers["연도"].min()),
    int(annual_passengers["연도"].max()) + 1
)

# 누락된 연도 추가
annual_passengers = (
    annual_passengers.set_index("연도")
    .reindex(all_years)
    .rename_axis("연도")
    .reset_index()
)

# 승객 수를 만 명 단위로 변환
annual_passengers["승객수_만명"] = (
    annual_passengers["승객수"] / 10_000
)

# 2026년은 누계로 표시
annual_passengers["연도표시"] = annual_passengers["연도"].apply(
    lambda year: "2026<br>(누계)" if year == 2026 else str(year)
)


# ============================================================
# 3. 색상 설정
# ============================================================

BLUE = "#3478C0"    # 파란색
ORANGE = "#E67E22"  # 오렌지색
GRAY = "#657487"    # 회색

# 2016년은 최댓값 강조, 2026년은 누계 데이터 구분
colors = [
    ORANGE if year == 2016
    else GRAY if year == 2026
    else BLUE
    for year in annual_passengers["연도"]
]


# ============================================================
# 4. 막대 위 숫자 표시
# ============================================================

labels = [
    f"{value:,.1f}" if pd.notna(value) else ""
    for value in annual_passengers["승객수_만명"]
]


# ============================================================
# 5. 그래프 생성
# ============================================================

fig_passengers = go.Figure()

fig_passengers.add_trace(
    go.Bar(
        x=annual_passengers["연도표시"],
        y=annual_passengers["승객수_만명"],

        marker_color=colors,
        marker_line_width=0,

        text=labels,
        textposition="outside",

        textfont=dict(
            size=20,
            color="#243247"
        ),

        cliponaxis=False,

        customdata=annual_passengers[
            ["연도", "승객수"]
        ].to_numpy(),

        hovertemplate=(
            "<b>%{customdata[0]:.0f}년</b>"
            "<br>승객수: %{customdata[1]:,.0f}명"
            "<extra></extra>"
        )
    )
)


# ============================================================
# 6. 제목 및 레이아웃
# ============================================================

fig_passengers.update_layout(
    title=dict(
        text=(
            "<b>부산항 크루즈 연도별 승객수</b>"
            "<br>"
            f"<span style='font-size:18px;color:{GRAY}'>"
            "연도별 승객 규모의 장기 변화"
            "</span>"
        ),

        # 제목과 부제목 중앙 정렬
        x=0.5,
        y=0.95,
        xanchor="center"
    ),

    template="plotly_white",

    width=1200,
    height=675,

    paper_bgcolor="white",
    plot_bgcolor="white",

    font=dict(
        family="Malgun Gothic, Arial, sans-serif",
        size=18,
        color="#243247"
    ),

    margin=dict(
        l=90,
        r=45,
        t=125,
        b=125
    ),

    bargap=0.32,
    showlegend=False
)


# ============================================================
# 7. X축 설정
# ============================================================

fig_passengers.update_xaxes(
    title=None,

    type="category",

    categoryorder="array",
    categoryarray=annual_passengers["연도표시"].tolist(),

    showgrid=False,

    tickfont=dict(size=17),

    fixedrange=True
)


# ============================================================
# 8. Y축 설정
# ============================================================

fig_passengers.update_yaxes(
    title="승객수 (만 명)",

    tickformat=",.0f",

    range=[
        0,
        annual_passengers["승객수_만명"].max() * 1.20
    ],

    gridcolor="#E9EDF2",
    zeroline=False,

    fixedrange=True
)


# ============================================================
# 9. 코로나19 영향 구간 표시
# ============================================================

year_list = annual_passengers["연도"].tolist()

if 2020 in year_list and 2022 in year_list:

    start = year_list.index(2020)
    end = year_list.index(2022)

    fig_passengers.add_vrect(
        x0=start - 0.5,
        x1=end + 0.5,

        fillcolor="#E85D5D",
        opacity=0.12,

        line_width=0,
        layer="below"
    )

    fig_passengers.add_annotation(
        x=(start + end) / 2,
        y=0.85,

        xref="x",
        yref="paper",

        text="<b>코로나19 영향 구간</b><br>(2020~2022년)",

        showarrow=False,
        align="center",

        font=dict(
            family="Malgun Gothic",
            size=18,
            color="#C62828"
        )
    )


# ============================================================
# 10. 그래프 하단 주석
# ============================================================

fig_passengers.add_annotation(
    x=0,
    y=-0.20,

    xref="paper",
    yref="paper",

    text=(
        "※ 2026년은 10월 16일까지 누계로, 이전 연도와 집계 기간이 다름"
        "<br>※ 승객 수는 확인된 값의 합계이며, 기록 없는 연도는 공백으로 표기함"
    ),

    showarrow=False,

    xanchor="left",
    align="left",

    font=dict(
        size=14,
        color=GRAY
    )
)


# ============================================================
# 11. 그래프 출력 및 이미지 저장 설정
# ============================================================

fig_passengers.show(
    config={
        "displaylogo": False,

        "toImageButtonOptions": {
            "format": "png",
            "filename": "PPT_연도별_승객수",
            "width": 1200,
            "height": 675,
            "scale": 3
        }
    }
)